# Part 3: Path Following

Follows the course MATLAB examples:
- **Path:** linear in joint space, `q(λ) = q_start·(1 − λ) + q_end·λ`, with λ from 0 to 1 (same as the MATLAB path script).
- **Trajectory:** `λ(t) = (Δλ/Δt)·t`. Every loop step commands the next point on the path, then waits `Δt`.
- **Measure:** read the actual joint angles back from the servos with the Dofbot software (`Arm_serial_servo_read`).
- **Error:** `q_desired(λ) − q(λ)` for each joint.

Keep this notebook in the same folder as `DofbotKinematics.ipynb` (used to plot the planned end effector path).

In [ ]:
%run DofbotKinematics.ipynb

import time
import matplotlib.pyplot as plt
from Arm_Lib import Arm_Device

arm = Arm_Device()

# Start with the arm straight up
arm.Arm_serial_servo_write6(90, 90, 90, 90, 90, 90, 2000)
time.sleep(2.5)

## Make a joint space path
Same as the MATLAB line `q(:,ii) = qstart*(1-lambda(ii)) + qend*lambda(ii)`.
`lam` is the list of λ values. Angles are rounded to whole degrees, since that's what the servos take.

In [ ]:
def joint_path(q_start, q_end, lam):
    path = [np.array(q_start) * (1 - l) + np.array(q_end) * l for l in lam]
    return np.round(path).astype(int)

## Plot the planned path (like the MATLAB path script)
Three plots along λ: the joint angles q(λ), the end effector orientation as ZYX Euler angles, and the end effector position P0T(λ) from `fk_Dofbot`.

In [ ]:
colors = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']

def plot_lines(x, y, labels, legend_title, xlabel, ylabel, title):
    plt.figure(figsize=(8.5, 3.5))
    for j in range(y.shape[1]):
        plt.plot(x, y[:, j], color=colors[j], linewidth=2, label=labels[j])
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(title=legend_title, frameon=False, loc='center left', bbox_to_anchor=(1.02, 0.5))
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


def plot_plan(q_start, q_end, lam, name):
    q = np.array([np.array(q_start) * (1 - l) + np.array(q_end) * l for l in lam])
    euler = np.array([fk_Dofbot(qi)[0].as_euler('ZYX', degrees=True) for qi in q])
    pos = np.array([fk_Dofbot(qi)[1] for qi in q])

    plot_lines(lam, q, ['1', '2', '3', '4', '5'], 'Joint', 'λ', 'q(λ) (deg)', f"{name}: joint angles")
    plot_lines(lam, euler, ['Z', 'Y', 'X'], 'Axis', 'λ', 'Euler angles (deg)', f"{name}: end effector orientation")
    plot_lines(lam, pos, ['x', 'y', 'z'], 'Axis', 'λ', 'P0T (m)', f"{name}: end effector position")

## Follow a path
1. Move to the start of the path and wait.
2. For each point: command it (the servo gets `Δt` to get there), wait `Δt`, then read back all 5 joint angles.

Returns the actual angles and the time of each reading.

The servos sometimes fail a read and return `None`, especially while moving. `read_angle` just asks again until it gets a real angle.

In [ ]:
def read_angle(i):
    a = arm.Arm_serial_servo_read(i)
    while a is None:                      # read failed, wait a moment and ask again
        time.sleep(0.01)
        a = arm.Arm_serial_servo_read(i)
    return a


def run_path(path, dt):
    # Go to the start of the path first
    arm.Arm_serial_servo_write6(*[int(a) for a in path[0]], 90, 2000)
    time.sleep(2.5)

    actual = []
    times = []
    t0 = time.time()
    for q in path:
        arm.Arm_serial_servo_write6(*[int(a) for a in q], 90, int(dt * 1000))
        time.sleep(dt)
        actual.append([read_angle(i) for i in range(1, 6)])
        times.append(time.time() - t0)
    return np.array(actual), np.array(times)

## Error summary and plot
Prints the RMS and max error for each joint, and plots the error along the path.

In [ ]:
def show_error(x, path, actual, title, xlabel="λ"):
    err = path - actual                      # q_desired - q_actual
    rms = np.sqrt(np.mean(err**2, axis=0))
    mx = np.max(np.abs(err), axis=0)

    print(title)
    print("RMS error per joint (deg):", np.round(rms, 2))
    print("Max error per joint (deg):", np.round(mx, 2))

    plot_lines(x, err, ['1', '2', '3', '4', '5'], 'Joint', xlabel, 'q_desired − q_actual (deg)', title)
    return err

## Run multiple paths
Path 1 is the example from the MATLAB path script, with joint 5 held at 90 (it stalls on small, slow moves).
Each path uses `N = 51` points (`Δλ = 0.02`) and `Δt = 0.1 s`.

In [ ]:
paths = [
    ([90, 100, 90, 90, 90],   [80, 135, 90, 110, 90]),
    ([45, 130, 60, 90, 90],   [135, 130, 60, 130, 90]),
    ([135, 130, 60, 130, 90], [30, 140, 30, 150, 90]),
]

N = 51
lam = np.linspace(0, 1, N)
dt = 0.1

for k, (q_start, q_end) in enumerate(paths):
    plot_plan(q_start, q_end, lam, f"Path {k+1}")
    path = joint_path(q_start, q_end, lam)
    actual, times = run_path(path, dt)
    print(f"\nTotal time: {times[-1]:.2f} s")
    show_error(lam, path, actual, f"Path {k+1}: {q_start} to {q_end}")

# 6000 Level: Trajectory Exploration

**Same path, two trajectories.** Path 2 from above, same total time `T = 3 s`, same loop time `Δt = 0.06 s`. Only the shape of λ(t) changes:
- **Linear:** `λ(t) = t / T`. Constant speed the whole way, so the arm starts and stops suddenly.
- **Trapezoidal:** `trapGen` from the course MATLAB, with `ta = 1 s` to speed up and 1 s to slow down. It starts and ends slowly, but has to go 1.5× faster in the middle to finish in the same time.

All joints share the same λ(t), like the MATLAB "same time indexing" case, so the arm stays on the path.

**Prediction:** the linear trajectory starts at full speed right away, so the error should jump up at the start and stay at a steady lag. The trapezoidal trajectory should have almost no error at the start and end, since it's moving slowly there, but a bigger lag in the middle where it's moving 1.5× faster. Joint 1 moves the most (90°), so it should show this the most.

In [ ]:
def trapGen(t, T, ta):
    # Same as the course MATLAB: lambda goes 0 to 1 with speed-up time ta and slow-down time ta
    v = 1 / (T - ta)
    a = v / ta
    s = np.zeros(len(t))
    for k in range(len(t)):
        if t[k] < ta:
            s[k] = 0.5 * a * t[k]**2
        elif t[k] <= T - ta:
            s[k] = v * (t[k] - ta / 2)
        else:
            s[k] = 1 - 0.5 * a * (T - t[k])**2
    return s


T = 3.0
dt_traj = 0.06
t = np.arange(0, T + dt_traj / 2, dt_traj)     # loop times

lam_linear = t / T
lam_trap = trapGen(t, T, 1.0)

plot_lines(t, np.column_stack([lam_linear, lam_trap]), ['Linear', 'Trapezoidal'], 'λ(t)', 't (s)', 'λ', 'Two timing profiles for the same path')

In [ ]:
q_start, q_end = paths[1]

path_linear = joint_path(q_start, q_end, lam_linear)
path_trap = joint_path(q_start, q_end, lam_trap)

actual_linear, times_linear = run_path(path_linear, dt_traj)
actual_trap, times_trap = run_path(path_trap, dt_traj)

err_linear = show_error(t, path_linear, actual_linear, "Linear λ(t)", "t (s)")
err_trap = show_error(t, path_trap, actual_trap, "Trapezoidal λ(t)", "t (s)")

## Compare the two trajectories
Total time, overall RMS error, error during the first and last second (where the speeds are most different), and the error at the last point.

In [ ]:
for name, err, times in [("Linear", err_linear, times_linear), ("Trapezoidal", err_trap, times_trap)]:
    e = np.linalg.norm(err, axis=1)          # size of the error at each step
    print(name)
    print("  Total time (s):", round(times[-1], 2))
    print("  Overall RMS error (deg):", round(np.sqrt(np.mean(err**2)), 2))
    print("  Mean error, first 1 s (deg):", round(np.mean(e[t < 1]), 2))
    print("  Mean error, middle (deg):", round(np.mean(e[(t >= 1) & (t <= 2)]), 2))
    print("  Mean error, last 1 s (deg):", round(np.mean(e[t > 2]), 2))
    print("  Error at end of path (deg):", err[-1])

plot_lines(t, np.column_stack([np.linalg.norm(err_linear, axis=1), np.linalg.norm(err_trap, axis=1)]),
           ['Linear', 'Trapezoidal'], 'λ(t)', 't (s)', '‖q_desired − q_actual‖ (deg)', 'Same path, two trajectories')